# 02 - Función de Costo (Log-Loss), Optimización y Fronteras de Decisión 2D

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno estudiaremos la mecanica interna de optimizacion de la **Regresion Logistica**. Analizaremos por que el error cuadratico medio (MSE) produce una superficie no convexa en clasificacion, deduciremos la funcion de **Entropia Cruzada Binaria (Log-Loss)** a partir del principio de Maxima Verosimilitud, calcularemos su gradiente analitico y visualizaremos la **frontera de decision lineal** y el mapa de contorno de probabilidades en dos dimensiones sobre el dataset de cancer de mama.

---

### Contenidos
1. No Convexidad del Error Cuadratico Medio en Clasificacion
2. Principio de Maxima Verosimilitud (MLE) y Deduccion de Log-Loss
3. Comportamiento y Penalizacion de la Funcion Log-Loss
4. Gradiente Analitico y Algoritmos de Optimizacion (L-BFGS, SAGA)
5. La Importancia del Escalado en la Convergencia del Optimizador
6. Modelo Bidimensional sobre `load_breast_cancer`
7. Ecuacion Algebraica de la Frontera de Decision Lineal
8. Visualizacion 2D: Superficie de Probabilidad y Frontera de Separacion
9. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss, accuracy_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. No Convexidad del Error Cuadrático Medio en Clasificación

Si intentamos utilizar el Error Cuadratico Medio (MSE) reemplazando la prediccion por la funcion sigmoide:

$$J_{MSE}(\beta) = \frac{1}{2n} \sum_{i=1}^n (y_i - \sigma(\beta^T x_i))^2$$

La composicion con la no linealidad sigmoidea introduce regiones donde la segunda derivada cambia de signo, produciendo una funcion **no convexa** repleta de minimos locales y zonas de gradiente plano (desvanecimiento del gradiente). Un optimizador por descenso de gradiente quedaria atrapado facilmente en soluciones suboptimas.


## 2. Principio de Máxima Verosimilitud (MLE) y Deducción de Log-Loss

Asumiendo que las observaciones $y_i$ siguen de manera independiente una distribucion de Bernoulli con probabilidad $\hat{p}_i = \sigma(\beta^T x_i)$:

$$P(Y = y_i \mid x_i) = \hat{p}_i^{y_i} (1 - \hat{p}_i)^{1 - y_i}$$

La funcion de **verosimilitud (Likelihood)** del conjunto de datos completo es el producto de las probabilidades individuales:

$$L(\beta) = \prod_{i=1}^n \hat{p}_i^{y_i} (1 - \hat{p}_i)^{1 - y_i}$$

Aplicando el logaritmo natural para convertir el producto en suma (Log-Likelihood):

$$\ell(\beta) = \sum_{i=1}^n \left[ y_i \ln(\hat{p}_i) + (1 - y_i) \ln(1 - \hat{p}_i) \right]$$

Maximizar la verosimilitud equivale a **minimizar la perdida negativa media**, conocida como **Entropia Cruzada Binaria (Binary Cross-Entropy)** o **Log-Loss**:

$$J(\beta) = -\frac{1}{n} \sum_{i=1}^n \left[ y_i \ln(\hat{p}_i) + (1 - y_i) \ln(1 - \hat{p}_i) \right]$$

Esta funcion de costo es **estrictamente convexa**, lo que garantiza que cualquier minimo local es a su vez el unico minimo global.


In [ ]:
# Visualizacion grafica de la penalizacion de Log-Loss
p = np.linspace(1e-4, 1 - 1e-4, 300)

costo_y1 = -np.log(p)       # Cuando la etiqueta real es y = 1
costo_y0 = -np.log(1 - p)   # Cuando la etiqueta real es y = 0

plt.figure(figsize=(11, 5))
plt.plot(p, costo_y1, color='#1f77b4', linewidth=2.5, label='Costo si y = 1:  -ln(p)')
plt.plot(p, costo_y0, color='#d62728', linewidth=2.5, label='Costo si y = 0:  -ln(1 - p)')

plt.axvline(0.5, color='gray', linestyle=':', alpha=0.7)
plt.title('Penalizacion de Log-Loss segun la Probabilidad Estimada', fontsize=13, fontweight='bold')
plt.xlabel('Probabilidad Estimada p = P(Y=1|X)', fontsize=11, fontweight='bold')
plt.ylabel('Costo Individual de la Observacion', fontsize=11, fontweight='bold')
plt.ylim(0, 5)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 3. Gradiente Analítico y Algoritmos de Optimización

Al diferenciar $J(\beta)$ con respecto al vector de parametros $\beta$, la derivada de la sigmoide $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ cancela exactamente los denominadores, produciendo un gradiente notablemente elegante:

$$\nabla_\beta J(\beta) = \frac{1}{n} X^T (\hat{p} - y)$$

Donde:
- $\hat{p} = [\hat{p}_1, \dots, \hat{p}_n]^T$: Vector de probabilidades predichas.
- $y = [y_1, \dots, y_n]^T$: Vector de etiquetas reales.

A diferencia de la regresion lineal, no existe una formula cerrada analitica para despejar $\beta$ debido a la presencia no lineal de $\sigma(X\beta)$. Se emplean metodos iterativos:
- **L-BFGS** (Limited-memory Broyden-Fletcher-Goldfarb-Shanno): Algoritmo quasi-Newton de convergencia superlineal, predeterminado en Scikit-Learn.
- **SAGA / Liblinear**: Adecuados para datasets masivos o cuando se anaden penalizaciones de regularizacion $L_1$.


## 4. Modelo Bidimensional sobre `load_breast_cancer`

Para visualizar geometricamente la superficie de probabilidad y la recta de separacion, seleccionaremos dos caracteristicas clinicas del tumor:
1. `mean radius` (Radio medio de los nucleos celulares).
2. `mean texture` (Textura media: desviacion estandar de los valores de escala de grises).


In [ ]:
cancer = load_breast_cancer(as_frame=True)
df = cancer.frame

vars_2d = ['mean radius', 'mean texture']
X_2d = df[vars_2d]
y = df['target'] # 0 = Maligno, 1 = Benigno

X_train, X_test, y_train, y_test = train_test_split(
    X_2d, y, test_size=0.20, random_state=42, stratify=y
)

# Pipeline con StandardScaler y LogisticRegression
pipeline_2d = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, solver='lbfgs'))
])

pipeline_2d.fit(X_train, y_train)

# Extraccion de coeficientes en escala estandarizada
scaler_step = pipeline_2d.named_steps['scaler']
clf_step = pipeline_2d.named_steps['clf']

b0 = clf_step.intercept_[0]
b1 = clf_step.coef_[0][0]
b2 = clf_step.coef_[0][1]

print(f"Parametros del modelo (en espacio estandarizado):")
print(f"  Intercepto (beta_0):               {b0:.4f}")
print(f"  Coeficiente mean radius  (beta_1): {b1:.4f}")
print(f"  Coeficiente mean texture (beta_2): {b2:.4f}")


## 5. Ecuación Algebraica de la Frontera de Decisión

La frontera de decision estandar clasifica como clase 1 cuando $\hat{p} \ge 0.5$, lo cual equivale a:

$$z = \beta_0 + \beta_1 z_1 + \beta_2 z_2 = 0$$

Donde $z_1 = \frac{x_1 - \mu_1}{\sigma_1}$ y $z_2 = \frac{x_2 - \mu_2}{\sigma_2}$ son las caracteristicas estandarizadas.

Despejando $z_2$ en funcion de $z_1$, obtenemos una linea recta en el plano:

$$z_2 = -\frac{\beta_1}{\beta_2} z_1 - \frac{\beta_0}{\beta_2}$$

Y transformando de regreso al espacio de las caracteristicas originales $(x_1, x_2)$:

$$x_2 = \sigma_2 \left( -\frac{\beta_1}{\beta_2} \left( \frac{x_1 - \mu_1}{\sigma_1} \right) - \frac{\beta_0}{\beta_2} \right) + \mu_2$$


## 6. Visualización 2D: Superficie de Probabilidad y Frontera de Separación

Construimos una malla densa bidimensional para mapear las probabilidades continuas $P(Y=1 \mid X)$ con curvas de nivel (`contourf`) y trazar la frontera lineal de decision en $p = 0.5$.


In [ ]:
# Definicion de limites del espacio original
x1_min, x1_max = X_2d['mean radius'].min() - 1, X_2d['mean radius'].max() + 1
x2_min, x2_max = X_2d['mean texture'].min() - 1, X_2d['mean texture'].max() + 1

xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 400),
                       np.linspace(x2_min, x2_max, 400))

# Evaluacion de probabilidades sobre la malla
grid_points = np.c_[xx1.ravel(), xx2.ravel()]
probs = pipeline_2d.predict_proba(grid_points)[:, 1].reshape(xx1.shape)

plt.figure(figsize=(12, 7))

# Mapa de contorno de probabilidades
contour = plt.contourf(xx1, xx2, probs, levels=25, cmap='RdBu_r', alpha=0.75)
cbar = plt.colorbar(contour)
cbar.set_label('Probabilidad Estimada de ser Benigno P(Y=1|X)', fontsize=11, fontweight='bold')

# Frontera de decision (p = 0.5) y lineas de equiprobabilidad (p = 0.25, p = 0.75)
line_05 = plt.contour(xx1, xx2, probs, levels=[0.5], colors=['black'], linewidths=[2.5], linestyles=['solid'])
line_other = plt.contour(xx1, xx2, probs, levels=[0.1, 0.25, 0.75, 0.9], colors=['gray'], linewidths=[1.0], linestyles=['dashed'])
plt.clabel(line_05, fmt='p = 0.5 (Frontera)', fontsize=10)
plt.clabel(line_other, fmt='p = %.2f', fontsize=9)

# Puntos de entrenamiento
idx_benigno = (y_train == 1)
idx_maligno = (y_train == 0)

plt.scatter(X_train.loc[idx_benigno, 'mean radius'], X_train.loc[idx_benigno, 'mean texture'],
            color='#1f77b4', edgecolor='k', s=50, label='Benigno (Clase 1)', alpha=0.85)
plt.scatter(X_train.loc[idx_maligno, 'mean radius'], X_train.loc[idx_maligno, 'mean texture'],
            color='#d62728', edgecolor='k', s=55, marker='s', label='Maligno (Clase 0)', alpha=0.85)

plt.xlabel('Radio Medio (mean radius)', fontsize=11, fontweight='bold')
plt.ylabel('Textura Media (mean texture)', fontsize=11, fontweight='bold')
plt.title('Frontera de Decision y Contornos de Probabilidad en Regresion Logistica 2D', fontsize=13, fontweight='bold')
plt.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

# Evaluacion de desempeno
y_pred_te = pipeline_2d.predict(X_test)
loss_te = log_loss(y_test, pipeline_2d.predict_proba(X_test)[:, 1])
acc_te = accuracy_score(y_test, y_pred_te)

print(f"Rendimiento en conjunto de prueba (2 caracteristicas):")
print(f"  Exactitud (Accuracy): {acc_te*100:.2f}%")
print(f"  Perdida Logaritmica (Log-Loss): {loss_te:.4f}")


## 7. Conclusiones

### Puntos Clave:
1. **Convexidad Garantizada**: La funcion Log-Loss se deriva formalmente mediante Maxima Verosimilitud y asegura la existencia de un unico optimo global.
2. **Gradiente Intuitivo**: La direccion de actualizacion $\frac{1}{n}X^T(\hat{p} - y)$ escala proporcionalmente con el error en la probabilidad asignada.
3. **Linealidad de la Frontera**: Aunque la probabilidad sea sigmoidal y curvada en el espacio de salida, la frontera de decision en el espacio de caracteristicas es un **hiperplano estrictamente lineal** ($z = 0$).
4. **Gradiente de Confianza**: La distancia perpendicular de una muestra a la frontera de decision determina directamente el grado de certeza del modelo.

En el siguiente cuaderno, **03_Evaluacion_Metricas_Clasificacion.ipynb**, aprenderemos por que la exactitud (accuracy) es insuficiente para evaluar diagnosticos medicos, y estudiaremos la matriz de confusion, precision, recall (sensibilidad), F1-Score, curvas ROC y curvas Precision-Recall.
